# Browserless: browser automation via MCP

[Browserless](https://browserless.io) is a hosted browser-automation MCP server. It exposes 10 tools — search, smart scraper, map, crawl, export, performance, function, download, and a multi-turn browser agent — over a single streamable-HTTP MCP endpoint. With `llama-index-tools-mcp` you can auto-import all 10 as LlamaIndex tools without writing a partner package.

Browser sessions are pinned by `Mcp-Session-Id`, so multi-turn workflows preserve browser state across calls. This notebook shows both stateless single-shot tools and the multi-turn `browserless_agent`.

## Setup

In [1]:
%pip install -q llama-index llama-index-tools-mcp llama-index-llms-anthropic

Note: you may need to restart the kernel to use updated packages.


### Credentials

Get a Browserless API token at [account.browserless.io](https://account.browserless.io). The agent example also uses Anthropic — set `ANTHROPIC_API_KEY` to run those cells.

In [2]:
import getpass
import os

if not os.environ.get("BROWSERLESS_TOKEN"):
    os.environ["BROWSERLESS_TOKEN"] = getpass.getpass("BROWSERLESS_TOKEN:\n")

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY:\n")

## Connect and list tools

Connect with `BasicMCPClient` and load the 10 tools as LlamaIndex tools. The default 30-second `timeout` is too short for the first call's MCP handshake plus browser warm-up — pass `timeout=120`.

In [ ]:
import logging

from llama_index.tools.mcp import BasicMCPClient, McpToolSpec

# Silence an MCP SDK pydantic validation warning on server progress
# notifications. Harmless and unrelated to the tool call results below.
logging.getLogger().addFilter(
    lambda r: "Failed to validate notification" not in r.getMessage()
)

client = BasicMCPClient(
    "https://mcp.browserless.io/mcp",
    headers={"Authorization": f"Bearer {os.environ['BROWSERLESS_TOKEN']}"},
    timeout=120,
)

tools = await McpToolSpec(client=client).to_tool_list_async()
for t in tools:
    print(t.metadata.name)

## Invoke a stateless tool directly

Eight of the 10 tools are stateless single-shot calls (`browserless_smartscraper`, `browserless_search`, `browserless_map`, `browserless_crawl`, `browserless_export`, `browserless_performance`, `browserless_function`, `browserless_download`). Each call is independent, so you can invoke them without an LLM in the loop.

In [ ]:
smartscraper = next(t for t in tools if t.metadata.name == "browserless_smartscraper")
result = await smartscraper.acall(url="https://example.com", formats=["markdown"])
print(str(result)[:400])

## Use stateless tools in an agent

Hand the stateless tools to a `FunctionAgent` powered by Claude. The model picks which tool to call at each step. This pattern works without any session-management gymnastics because every step is a single self-contained call.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.anthropic import Anthropic

stateless_tools = [
    t for t in tools
    if t.metadata.name not in ("browserless_agent", "browserless_skill")
]

agent = FunctionAgent(
    name="BrowserlessAgent",
    description="Browses and scrapes the web via the Browserless MCP server.",
    llm=Anthropic(model="claude-sonnet-4-6"),
    tools=stateless_tools,
    system_prompt="You are a research assistant. Use the Browserless tools to scrape and summarize the web.",
)

response = await agent.run(
    "Scrape https://news.ycombinator.com and list the top 5 headlines as a markdown bullet list."
)
print(response)

## Multi-turn browser agent

The `browserless_agent` tool is different: it's a stateful, multi-turn driver where each call (`goto`, `snapshot`, `click`, `type`, `text`, `evaluate`, ...) operates on a persistent browser session. To preserve that session across calls you need a single MCP `ClientSession` for the whole flow.

**The default `McpToolSpec.to_tool_list_async()` + `tool.acall(...)` pattern does not preserve state.** `BasicMCPClient.call_tool()` opens a fresh `_run_session()` per invocation, with a fresh `Mcp-Session-Id`. That's fine for the stateless tools above, but a multi-turn `browserless_agent` flow gets routed to a different browser on every call — your `goto` lands on browser A, your `snapshot` lands on browser B, and you get back `about:blank`.

To share state, use the `client._run_session()` async context manager directly and call `session.call_tool(...)` inside it. The method is underscore-prefixed (Python convention for non-public) but it's just an `@asynccontextmanager`. Until upstream ships a public `client.session()` API, this is the cleanest way to keep one MCP session alive across multiple tool calls.

In [ ]:
async with client._run_session() as session:
    await session.call_tool(
        "browserless_agent",
        arguments={"method": "goto", "params": {"url": "https://example.com"}},
    )
    await session.call_tool("browserless_agent", arguments={"method": "snapshot"})
    result = await session.call_tool(
        "browserless_agent",
        arguments={"method": "text", "params": {"selector": "h1"}},
    )
    print(result.content[0].text)

## Drive the multi-turn agent from an LLM

Wrap session-bound `call_tool` invocations in a `FunctionTool` the LLM can call. The whole agent run lives inside one `client._run_session()` block so every step shares the same browser.

In [ ]:
from llama_index.core.tools import FunctionTool

async with client._run_session() as session:
    async def browser_action(method: str, params: dict | None = None) -> str:
        """Drive the Browserless multi-turn browser agent. Methods include goto, snapshot, click, type, text, evaluate. Pass params per the method's schema (e.g. {\"url\": \"...\"} for goto, {\"selector\": \"...\"} for text)."""
        result = await session.call_tool(
            "browserless_agent",
            arguments={"method": method, "params": params or {}},
        )
        return result.content[0].text

    browser_tool = FunctionTool.from_defaults(
        async_fn=browser_action,
        name="browserless_agent",
        description="Drive a stateful browser. Use methods like goto, snapshot, click, type, text, evaluate. Snapshot before extracting elements so you know what selectors are available.",
    )

    browser_agent = FunctionAgent(
        name="BrowserlessBrowserAgent",
        description="Drives a real browser to navigate, click, and extract content.",
        llm=Anthropic(model="claude-sonnet-4-6"),
        tools=[browser_tool],
        system_prompt=(
            "You drive a real browser via the browserless_agent tool. "
            "Always snapshot the page before extracting elements so you know what selectors are available. "
            "Use selector references from the snapshot when calling text/click."
        ),
    )

    response = await browser_agent.run(
        "Navigate to https://news.ycombinator.com, click the first story link, and report the title and first paragraph of that page."
    )
    print(response)

## Resources

- Hosted MCP server: [mcp.browserless.io](https://mcp.browserless.io)
- Runnable cookbook with all examples above: [`browserless/browserless-llamaindex`](https://github.com/browserless/browserless-llamaindex)
- LlamaIndex MCP adapter: [`llama-index-tools-mcp`](https://github.com/run-llama/llama_index/tree/main/llama-index-integrations/tools/llama-index-tools-mcp)